# Interview questions, and a second look at our own reasoning

Two more things, both grounded in the evaluation from notebook 02, not
generic: `generate_questions` writes interview questions that target this
candidate's specific gaps and transferable-skill claims. `check_bias`
rereads the evaluation's own reasoning for language that leans on something
other than job performance - and flags it, it does not silently fix it.

## Step 1 - the same setup as notebook 02

In [ ]:
import os, getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("LITELLM_API_KEY"):
    os.environ["LITELLM_API_KEY"] = getpass.getpass("LiteLLM API key: ")

from recruiting import (
    extract_candidate,
    extract_job_description,
    evaluate_candidate,
    rank_candidates,
    generate_questions,
    apply_bias_check,
    check_bias,
)
from recruiting.models import CandidateEvaluation, EvidenceMatch

SAMPLE_DIR = Path("sample_data")
jd = extract_job_description(
    (SAMPLE_DIR / "job_description.txt").read_text(encoding="utf-8"), source_file="job_description.txt"
)
resume_files = sorted(SAMPLE_DIR.glob("resume_*.txt"))
candidates = [extract_candidate(p.read_text(encoding="utf-8"), source_file=p.name) for p in resume_files]
ranked = rank_candidates([evaluate_candidate(c, jd) for c in candidates])
print("Ranked:", [(e.candidate.name, e.score) for e in ranked])

## Step 2 - interview questions for the top candidate

In [2]:
top = ranked[0]
questions = generate_questions(top)
for q in questions:
    print("Q:", q.question)
    print("   why:", q.rationale, "| targets:", q.targets, "\n")

Q: You've worked with both Flask and Django for building REST APIs. Can you walk us through a specific example where you chose one framework over the other, and what trade-offs you considered in terms of scalability, maintainability, and team productivity?
   why: While the candidate demonstrates proficiency in both frameworks, understanding their decision-making process reveals depth of knowledge beyond surface-level familiarity. This probes whether they can make informed architectural choices and understand framework trade-offs—critical for a backend engineer who may need to guide technical decisions. | targets: Verifying strong proficiency in Python and REST API design; assessing architectural thinking and framework selection rationale 

Q: You introduced Celery for async settlement processing at PayFlow Egypt. Walk us through the problem you were solving, how you designed the task queue architecture, and what challenges you encountered with distributed task processing, retries, or 

## Step 3 - checking every evaluation for bias

In [3]:
for e in ranked:
    apply_bias_check(e)
    print(e.candidate.name, "->", e.bias_flags or "no flags")

Amina Hassan -> no flags


Karim El-Sayed -> no flags


Lina Farouk -> no flags


## Step 4 - seeing it catch something

Our sample resumes are clean, so the check above mostly comes back empty -
good, that is what we want. To see the flag itself fire, feed it a
deliberately bad piece of reasoning.

In [4]:
biased_example = CandidateEvaluation(
    candidate=top.candidate,
    job_title=jd.title,
    score=70,
    matches=[
        EvidenceMatch(
            requirement="Strong proficiency in Python",
            essential=True,
            matched=True,
            evidence="5 years of Python experience",
            transferable=False,
            note="Graduated in 2019, so still young enough to be comfortable with modern frameworks.",
        )
    ],
)
print(check_bias(biased_example))

["Age bias: 'still young enough' implies age-based assumptions about technical capability rather than evaluating actual framework proficiency", 'Graduation year used as proxy: Graduation year (2019) is used to infer technical skills rather than assessing demonstrated competency with specific frameworks']


## What just happened

Questions that reference this candidate's actual gaps, not a generic
checklist. A bias check that reads the evaluation's *reasoning*, not the
candidate's resume - because that is where bias can creep in even after
`extract.py` already leaves irrelevant personal facts out.

Your turn: write your own biased-sounding note and see whether `check_bias`
catches it. Try something subtler than the age example above, e.g. a
comment about "culture fit" with no connection to any requirement.